# 03 · Metrics + scikit-learn (ช่วงที่ 3)

> **เป้าหมาย:** อ่าน confusion matrix เป็น, รู้ว่าทำไม accuracy อย่างเดียวหลอกได้, คำนวณ precision / recall / F1 / macro-F1 ได้ทั้งด้วยมือและด้วย sklearn, และรู้วิธีทำให้โมเดล "ไม่แน่ใจ" ได้ (ตรงกับข้อกำหนดโปรเจกต์ที่ให้แสดงค่าความมั่นใจ)

**เวลาที่ใช้:** ~1 ชม. | ต้องใช้: numpy, matplotlib, scikit-learn

| ส่วน | หัวข้อ |
|---|---|
| 1 | Accuracy หลอกได้อย่างไร |
| 2 | Confusion matrix 2 คลาส: TP / FP / FN / TN |
| 3 | Precision, Recall, F1 (คำนวณมือแล้วเทียบ sklearn) + ผลของ threshold |
| 4 | หลายคลาส: confusion matrix 10 คลาส, macro-F1 vs accuracy |
| 5 | ความมั่นใจ (confidence) และการตอบว่า "ไม่แน่ใจ" |
| 6 | sklearn workflow: fit / predict / Pipeline / cross-validation / latency |
| 7 | สรุป + resources |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import time, warnings; warnings.filterwarnings("ignore")
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, confusion_matrix,
                             classification_report, ConfusionMatrixDisplay, precision_recall_curve)
plt.rcParams["figure.dpi"] = 110
rng = np.random.default_rng(0)

## 1. Accuracy หลอกได้อย่างไร

`accuracy = ทายถูก ÷ ทายทั้งหมด` ฟังดูพอ แต่ถ้าข้อมูล **ไม่สมดุล** จะหลอก

ตัวอย่าง: ตรวจหาของเสียในโรงงาน ของเสียมีแค่ 5% ลองสร้าง "โมเดลโง่" ที่ตอบ **"ปกติ"** ทุกครั้งโดยไม่ดูข้อมูลเลย

In [ ]:
from sklearn.dummy import DummyClassifier
y = (rng.random(1000) < 0.05).astype(int)          # 1 = ของเสีย (~5%)
X_dummy = np.zeros((1000, 1))
dumb = DummyClassifier(strategy="most_frequent").fit(X_dummy, y)
pred = dumb.predict(X_dummy)

print("จำนวนของเสียจริง:", y.sum(), "จาก", len(y))
print("accuracy       :", accuracy_score(y, pred))
print("recall (จับของเสียได้กี่ %):", recall_score(y, pred))

accuracy ~95% แต่ **ไม่เคยจับของเสียได้เลย** (recall = 0) โมเดลไร้ประโยชน์ทั้งที่ตัวเลขสวย

**เกี่ยวกับโปรเจกต์ภาษามือ:** ถ้าบางคำมีตัวอย่างน้อยกว่าคำอื่นมาก โมเดลอาจ "ละเลย" คำหายากแล้ว accuracy ก็ยังดูดี นี่คือเหตุผลที่โจทย์กำหนดให้ใช้ **macro-F1 และ confusion matrix** ควบคู่ accuracy

> 🧪 **Mini challenge 1.1:** เปลี่ยนสัดส่วน `0.05` เป็น `0.5` accuracy ของโมเดลโง่เป็นเท่าไร? (ทำไมถึงเป็นอย่างนั้น?)

## 2. Confusion matrix (2 คลาส)

ตารางที่ไล่ดูว่าโมเดลทำนาย *อะไร* เมื่อความจริงคือ *อะไร* (แถว = ความจริง, คอลัมน์ = ที่ทำนาย — ตามรูปแบบของ sklearn)

```
                    ทำนาย: ไม่ใช่ (0)     ทำนาย: ใช่ (1)
ความจริง: ไม่ใช่ (0)   TN (ถูก)             FP (ทายว่าใช่ แต่จริงไม่ใช่ → "เตือนผิด")
ความจริง: ใช่ (1)      FN (ทายว่าไม่ใช่ แต่จริงใช่ → "พลาด")   TP (ถูก)
```
จำคำง่ายๆ: ตัวอักษรหลัง (P/N) = สิ่งที่โมเดล *ทำนาย* | ตัวอักษรแรก (T/F) = ทายถูกหรือผิด

In [ ]:
# ตัวอย่างขนาดเล็กที่คำนวณด้วยมือได้: 10 ตัวอย่าง
y_true = np.array([1, 1, 1, 1, 0, 0, 0, 0, 0, 0])
y_pred = np.array([1, 1, 1, 0, 0, 0, 0, 0, 1, 1])

cm = confusion_matrix(y_true, y_pred)
print(cm, "\n")
TN, FP, FN, TP = cm.ravel()
print(f"TP={TP}  FP={FP}  FN={FN}  TN={TN}")

fig, ax = plt.subplots(figsize=(4.4, 3.8))
ConfusionMatrixDisplay(cm, display_labels=["0: no", "1: yes"]).plot(ax=ax, cmap="Blues", colorbar=False)
ax.set_title("rows = truth, columns = prediction"); plt.show()

**ดูให้ออก:** แนวทแยงหลัก (TN, TP) = ถูก นอกทแยง = ผิด และ *ผิดคนละแบบ*: FP กับ FN มีต้นทุนต่างกันในแต่ละงาน (ตรวจมะเร็ง FN อันตรายกว่า; กรองสแปม FP อันตรายกว่าเพราะอีเมลสำคัญหาย)

## 3. Precision, Recall, F1

| เมตริก | สูตร | ตอบคำถามว่า |
|---|---|---|
| **Precision** | TP / (TP + FP) | "ที่โมเดลบอกว่าใช่ **จริงๆ แล้วใช่กี่ %**" (เชื่อได้แค่ไหนเวลามันร้อง) |
| **Recall** (= sensitivity) | TP / (TP + FN) | "ของที่เป็นใช่จริงๆ **โมเดลจับได้กี่ %**" (พลาดไปเท่าไร) |
| **F1** | 2·P·R / (P + R) | ค่าเฉลี่ยแบบ harmonic ของสองตัว — ต่ำถ้าตัวใดตัวหนึ่งต่ำ |
| **Specificity** | TN / (TN + FP) | recall ของคลาส "ไม่ใช่" |

In [ ]:
precision = TP / (TP + FP); recall = TP / (TP + FN); f1 = 2 * precision * recall / (precision + recall)
specificity = TN / (TN + FP)
print(f"คำนวณมือ : precision={precision:.3f}  recall={recall:.3f}  F1={f1:.3f}  specificity={specificity:.3f}")
print(f"sklearn  : precision={precision_score(y_true,y_pred):.3f}  recall={recall_score(y_true,y_pred):.3f}  F1={f1_score(y_true,y_pred):.3f}")
print("accuracy :", accuracy_score(y_true, y_pred), "(= (TP+TN)/ทั้งหมด =", (TP + TN) / len(y_true), ")")

### ผลของ threshold: precision กับ recall แลกกัน
โมเดลส่วนใหญ่ให้ **ความน่าจะเป็น** ก่อน แล้วเราค่อยตัดสินว่า "เกินกี่ % ถึงจะตอบว่าใช่" (threshold) เลื่อน threshold = เปลี่ยนจุดสมดุลระหว่าง precision กับ recall

In [ ]:
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split

Xb, yb = make_classification(n_samples=4000, n_features=8, n_informative=4, weights=[0.85, 0.15], flip_y=0.01, class_sep=1.2, random_state=3)
Xtr, Xte, ytr, yte = train_test_split(Xb, yb, test_size=0.3, random_state=0, stratify=yb)
proba = LogisticRegression(max_iter=1000).fit(Xtr, ytr).predict_proba(Xte)[:, 1]

ths = np.linspace(0.05, 0.95, 91)
P = [precision_score(yte, proba >= t, zero_division=1) for t in ths]
R = [recall_score(yte, proba >= t) for t in ths]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(ths, P, label="precision"); axes[0].plot(ths, R, label="recall"); axes[0].plot(ths, [f1_score(yte, proba >= t) for t in ths], "--", label="F1")
axes[0].set_xlabel("threshold"); axes[0].legend(); axes[0].grid(alpha=0.3); axes[0].set_title("Raise threshold: precision up, recall down")
for t, c in [(0.2, "#f59e0b"), (0.5, "#2563eb"), (0.8, "#dc2626")]:
    axes[1].scatter(recall_score(yte, proba >= t), precision_score(yte, proba >= t, zero_division=1), s=90, c=c, label=f"threshold={t}")
pr, rc, _ = precision_recall_curve(yte, proba); axes[1].plot(rc, pr, "gray", alpha=0.5)
axes[1].set_xlabel("recall"); axes[1].set_ylabel("precision"); axes[1].legend(); axes[1].set_title("Precision-recall curve"); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

**สังเกต:** threshold ต่ำ → โมเดลตอบ "ใช่" บ่อย → จับได้เยอะ (recall สูง) แต่เตือนผิดเยอะ (precision ต่ำ); threshold สูง → ตอบ "ใช่" เฉพาะมั่นใจมาก → precision สูง แต่พลาดเยอะ **ไม่มีค่าที่ดีสุดทุกงาน** ต้องเลือกตามต้นทุนของ FP vs FN

> 🧪 **Mini challenge 3.1:** จากกราฟ ถ้าต้องการ recall ≥ 0.8 ควรตั้ง threshold ประมาณเท่าไร และ precision เหลือเท่าไร? (ลองเขียนโค้ดหาเอง)
>
> ❓ **คิดดู:** สำหรับแอปรู้จำภาษามือที่ใช้สื่อสารจริง คำตอบผิดแบบ "มั่นใจแต่ผิด" กับ "ไม่ตอบ" อย่างไหนแย่กว่ากัน?

## 4. หลายคลาส: confusion matrix 10 คลาส, macro-F1

โปรเจกต์ของเรามีหลายคำ (คลาส) ใช้ชุดข้อมูลตัวเลขลายมือ `load_digits` ของ sklearn เป็นตัวแทน (ภาพ 8x8 → คลาส 0-9) เพราะเป็นข้อมูลจริง เล็ก และรันได้ทันที ไม่ต้องต่อเน็ต

In [ ]:
from sklearn.datasets import load_digits
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

digits = load_digits()
Xd, yd = digits.data, digits.target
print("ข้อมูล:", Xd.shape, "| คลาส:", np.unique(yd))

fig, axes = plt.subplots(1, 8, figsize=(12, 1.9))
for ax, img, lab in zip(axes, digits.images[:8], yd[:8]):
    ax.imshow(img, cmap="gray_r"); ax.set_title(f"label {lab}", fontsize=9); ax.axis("off")
plt.show()

Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3, random_state=0, stratify=yd)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
pred = clf.predict(Xte)
print("accuracy:", round(accuracy_score(yte, pred), 4), "| macro-F1:", round(f1_score(yte, pred, average="macro"), 4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
ConfusionMatrixDisplay.from_predictions(yte, pred, ax=axes[0], cmap="Blues", colorbar=False); axes[0].set_title("Counts")
ConfusionMatrixDisplay.from_predictions(yte, pred, normalize="true", ax=axes[1], cmap="Blues", colorbar=False, values_format=".2f"); axes[1].set_title("Normalized by row (= recall per class)")
plt.tight_layout(); plt.show()

# หาคู่คลาสที่สับสนกันมากที่สุด
cm = confusion_matrix(yte, pred); off = cm.copy(); np.fill_diagonal(off, 0)
pairs = sorted([(off[i, j], i, j) for i in range(10) for j in range(10) if off[i, j] > 0], reverse=True)[:5]
print("คู่ที่สับสนมากสุด (จริง -> ทำนาย : จำนวน):")
for n, i, j in pairs: print(f"  {i} -> {j} : {n}")

### วิธีอ่าน confusion matrix หลายคลาส
1. **ทแยงมุมหลัก** ควรเข้มที่สุด
2. **จุดสว่างนอกทแยง** = คู่ที่สับสนกัน (อ่านแถว → คอลัมน์: "ความจริงคือแถว แต่ทายเป็นคอลัมน์") ในงานภาษามือ ข้อมูลตรงนี้บอกว่า **คำไหนมีท่าคล้ายกัน** ซึ่งเป็นสิ่งที่โจทย์ให้ "วิเคราะห์คำที่สับสนกัน"
3. แบบ **normalized by row** ทำให้เทียบคลาสที่มีจำนวนตัวอย่างต่างกันได้ ค่าบนทแยง = recall ของคลาสนั้น

### Accuracy vs macro-F1 vs weighted-F1
| average | วิธีรวม | เมื่อไร |
|---|---|---|
| **micro** (≈ accuracy ใน single-label) | รวม TP/FP/FN ทุกคลาสก่อนคำนวณ | ดูภาพรวมตามจำนวนตัวอย่าง |
| **macro** | หา F1 แต่ละคลาส แล้วเฉลี่ย *เท่ากัน* ทุกคลาส | **ทุกคำสำคัญเท่ากัน — คำหายากไม่ถูกกลบ** |
| **weighted** | เฉลี่ยถ่วงน้ำหนักด้วยจำนวนตัวอย่าง | คล้าย accuracy |

ทดลอง: ทำให้คลาส 9 **หายากมาก** แล้วดูว่า accuracy กับ macro-F1 ตอบสนองต่างกันอย่างไร

In [ ]:
keep = np.ones(len(yd), bool)
nine = np.where(yd == 9)[0]; keep[nine[8:]] = False                 # เหลือคลาส 9 แค่ 8 ตัวอย่าง
Xi, yi = Xd[keep], yd[keep]
print("จำนวนตัวอย่างแต่ละคลาส:", np.bincount(yi))

Xtr, Xte, ytr, yte = train_test_split(Xi, yi, test_size=0.3, random_state=0, stratify=yi)
clf_i = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
pred_i = clf_i.predict(Xte)

print(f"\naccuracy    = {accuracy_score(yte, pred_i):.3f}")
print(f"macro-F1    = {f1_score(yte, pred_i, average='macro'):.3f}")
print(f"weighted-F1 = {f1_score(yte, pred_i, average='weighted'):.3f}")
print("\nclassification_report:")
print(classification_report(yte, pred_i, digits=3))

**อ่านผล:** ดูแถว `9` ใน report (precision / recall / **support** = จำนวนตัวอย่างจริงในชุดทดสอบ) เมื่อคลาสหายากมี support น้อยมาก ตัวเลขของคลาสนั้นผันผวนสูง (ตัวอย่างเดียวผิดเปลี่ยน recall หลายสิบ %) และ **macro-F1 ไวต่อคลาสนั้นมากกว่า accuracy** เพราะทุกคลาสมีน้ำหนักเท่ากัน

⚠️ ข้อสังเกตเชิงวิธีการ: ถ้า support ของคลาสหนึ่งมีแค่ 2-3 ตัวอย่าง อย่าเชื่อตัวเลขของมันมากเกินไป — นี่คือเหตุผลที่โจทย์ย้ำเรื่อง *ชุดข้อมูลที่มีขนาดเพียงพอ* และการดูผลแยกคลาส

> 🧪 **Mini challenge 4.1:** ลองเปลี่ยน `nine[8:]` เป็น `nine[3:]` (เหลือ 3 ตัวอย่าง) แล้วดู recall ของคลาส 9 และ macro-F1 ที่เปลี่ยนไป
>
> 🧪 **Mini challenge 4.2:** ใช้ `LogisticRegression(class_weight="balanced")` แล้วเทียบ recall ของคลาส 9 `class_weight` ทำอะไรกับการฝึก? (คำใบ้: ถ่วงความผิดพลาดของคลาสหายากให้หนักขึ้น)

## 5. ความมั่นใจ (confidence) และการตอบว่า "ไม่แน่ใจ"

ข้อกำหนดโปรเจกต์: *"แสดงผลคำที่คาดการณ์และค่าความมั่นใจ พร้อมรองรับกรณีที่โมเดลไม่มั่นใจ"*

โมเดลส่วนใหญ่ให้ความน่าจะเป็นของแต่ละคลาส (`predict_proba`) ค่าสูงสุดคือ confidence ถ้าต่ำกว่า threshold → ตอบ "ไม่แน่ใจ" แลกกับว่า **ตอบน้อยลง (coverage ลด)** แต่ **ที่ตอบแม่นขึ้น**

In [ ]:
Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.3, random_state=0, stratify=yd)
clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)).fit(Xtr, ytr)
P_all = clf.predict_proba(Xte); conf = P_all.max(1); pred = P_all.argmax(1)

thresholds = np.linspace(0.3, 0.999, 60)
coverage = [(conf >= t).mean() for t in thresholds]
acc_answered = [accuracy_score(yte[conf >= t], pred[conf >= t]) if (conf >= t).sum() else np.nan for t in thresholds]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(conf[pred == yte], bins=25, alpha=0.7, label="correct predictions"); axes[0].hist(conf[pred != yte], bins=25, alpha=0.8, label="wrong predictions")
axes[0].set_xlabel("confidence"); axes[0].legend(); axes[0].set_title("Wrong predictions tend to have lower confidence")
axes[1].plot(thresholds, coverage, label="coverage (fraction answered)"); axes[1].plot(thresholds, acc_answered, label="accuracy on answered")
axes[1].set_xlabel("confidence threshold"); axes[1].legend(); axes[1].grid(alpha=0.3); axes[1].set_title("Abstaining trades coverage for accuracy")
plt.tight_layout(); plt.show()

for t in [0.5, 0.8, 0.95]:
    m = conf >= t; print(f"threshold {t}: ตอบ {m.mean():.1%} ของทั้งหมด | แม่นในกลุ่มที่ตอบ {accuracy_score(yte[m], pred[m]):.1%} | 'ไม่แน่ใจ' {(~m).sum()} จาก {len(m)}")

**สังเกต:** คำตอบที่ผิดมักมี confidence ต่ำกว่า (แต่ **ไม่ใช่เสมอไป** — โมเดลก็มั่นใจผิดได้) การตั้ง threshold คือการตัดสินใจเรื่องการออกแบบ: ยิ่งสูง ยิ่งแม่นแต่ตอบน้อย

> ⚠️ **ข้อควรระวัง:** ความน่าจะเป็นจากโมเดลไม่ได้ "calibrated" โดยอัตโนมัติ (0.9 ไม่ได้แปลว่าถูก 90% จริงๆ เสมอไป) โดยเฉพาะโครงข่ายประสาทมักมั่นใจเกินจริง ต้องตรวจกับ validation ก่อนเชื่อ
>
> 🧪 **Mini challenge 5.1:** ถ้าแอปต้องการคำตอบที่ผิดไม่เกิน 1% จะตั้ง threshold เท่าไร? ตอบได้กี่ % ของครั้งทั้งหมด? (เขียนโค้ดหา)

## 6. sklearn workflow

รูปแบบที่ sklearn ทุกโมเดลใช้เหมือนกัน เรียนครั้งเดียวใช้ได้ทั้งหมด:

```
model = SomeModel(hyperparameters)     # 1. สร้าง
model.fit(X_train, y_train)            # 2. ฝึก (เรียนจาก train เท่านั้น!)
y_pred = model.predict(X_test)         # 3. ทำนาย
model.score(X_test, y_test)            # 4. วัดผล
```

**Pipeline** ต่อขั้นตอนเตรียมข้อมูล + โมเดลเข้าด้วยกัน ป้องกัน *leakage* เช่นการคำนวณ mean/std ของ scaler จาก test โดยไม่ตั้งใจ (scaler จะ `fit` เฉพาะข้อมูลฝึกเสมอ)

In [ ]:
from sklearn.model_selection import cross_val_score, GroupKFold, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier

pipe = make_pipeline(StandardScaler(), KNeighborsClassifier(5))
scores = cross_val_score(pipe, Xd, yd, cv=StratifiedKFold(5, shuffle=True, random_state=0))
print("5-fold CV accuracy:", scores.round(3), "| mean =", scores.mean().round(3), "| std =", scores.std().round(3))

# GroupKFold: ข้อมูลที่ "กลุ่ม" เดียวกันต้องอยู่ฝั่งเดียวกัน (ในโปรเจกต์: group = ผู้แสดง)
groups = np.arange(len(yd)) // 150                    # สมมติว่าแบ่งเป็นกลุ่มตามลำดับ (ตัวอย่างประกอบ ไม่ใช่ signer จริง)
scores_g = cross_val_score(pipe, Xd, yd, cv=GroupKFold(4), groups=groups)
print("GroupKFold accuracy:", scores_g.round(3), "| mean =", scores_g.mean().round(3))
print("\n(ตัวเลขสองแบบต่างกัน เพราะวิธีแบ่งต่างกัน — แบบ group เข้มงวดกว่า)")

# latency: โจทย์ให้วัดบนอุปกรณ์เป้าหมาย
sample = Xte[:1]; N = 300; pipe.fit(Xtr, ytr)
t0 = time.perf_counter(); [pipe.predict(sample) for _ in range(N)]; dt = (time.perf_counter() - t0) / N * 1000
print(f"inference latency ต่อ 1 ตัวอย่าง ≈ {dt:.2f} ms (เครื่องที่รัน notebook นี้เท่านั้น ไม่ใช่ตัวเลขของอุปกรณ์เป้าหมายของเธอ)")

**สังเกต:** ค่า CV เป็นช่วงของค่า ไม่ใช่ค่าเดียว (mean ± std) บอกว่าผลนิ่งแค่ไหน, `GroupKFold` คือเครื่องมือสำหรับแบ่ง "ตามผู้แสดง" ในโปรเจกต์จริง

> 🧪 **Mini challenge 6.1:** เปลี่ยน `KNeighborsClassifier(5)` เป็น `KNeighborsClassifier(1)` และ `(25)` แล้วเทียบ CV accuracy (k เล็ก = โมเดลยืดหยุ่น เสี่ยง overfit)
>
> 🧪 **Mini challenge 6.2:** ลบ `StandardScaler()` ออกจาก pipeline แล้วดูผล ทำไม KNN ถึงไวต่อ scaling?

## 7. สรุป + Resources

**ที่ควรอธิบายได้หลังจบ**
1. accuracy หลอกเมื่อข้อมูลไม่สมดุล → ดู precision/recall/F1 และ macro-F1
2. confusion matrix: แถว = ความจริง คอลัมน์ = ที่ทำนาย; นอกทแยง = ความสับสนที่ต้องวิเคราะห์
3. precision (เชื่อได้ไหม) vs recall (พลาดไหม) แลกกันด้วย threshold
4. macro-F1 ให้ทุกคลาสน้ำหนักเท่ากัน — เหมาะกับงานที่ทุกคำสำคัญเท่ากัน
5. ความมั่นใจ + ตอบ "ไม่แน่ใจ" = แลก coverage กับ accuracy
6. `Pipeline` กัน leakage, `GroupKFold` แบ่งตามกลุ่ม (ผู้แสดง)

### ควรอ่าน/ดูเพิ่ม (ค้นยืนยันแล้ว)
| หัวข้อ | แหล่ง |
|---|---|
| วิดีโอ StatQuest: *The Confusion Matrix*, *Sensitivity and Specificity*, *Cross Validation* | อยู่ใน playlist: https://www.classcentral.com/course/youtube-machine-learning-90293 |
| Google MLCC — Classification (thresholding, confusion matrix, precision, recall, AUC) | https://developers.google.com/machine-learning/crash-course |
| เอกสาร sklearn: `classification_report` (อธิบาย macro/weighted average) | https://scikit-learn.org/stable/modules/generated/sklearn.metrics.classification_report.html |
| sklearn User Guide — Model evaluation | https://scikit-learn.org/stable/modules/model_evaluation.html |
| บทเรียน notebook ครบชุด (sklearn, matplotlib) | https://github.com/jakevdp/PythonDataScienceHandbook |

> ⏭️ **ต่อไป:** `04_deep_learning_basics.ipynb` — neuron → network, loss, backprop, Adam, learning rate, batch size, epoch, อ่าน loss curve